In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [4]:
# Load dataset
data = pd.read_csv("D:/Sanctum/AI-ML-JOURNEY/Deep Learning/Data/Churn_Modelling.csv")

In [5]:
# Droping Irrelevant columns
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)

# Handling gender categorical feature
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# Handling Geography categorical column
ohe_geography = OneHotEncoder()
geo_ohe = ohe_geography.fit_transform(data[['Geography']]).toarray() # returns 2d array/df
geo_encoded_df = pd.DataFrame(geo_ohe,columns=ohe_geography.get_feature_names_out(['Geography'])) # columns name
# * combining with the original df
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

# Dividing the data into independent and dependent feature
X = data.drop('Exited',axis=1)
y = data['Exited']

# train-test split
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.20, random_state=42)
# Scale down 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
# Saving encoder to pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open("ohe_geography.pkl",'wb') as file:
    pickle.dump(ohe_geography,file)
# saving scaler to pickle
with open("scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)

In [13]:
# Define a function to create the model and try different parameters (KerasClassifier)
def create_model(neurons=32,layers=1):
    # Default
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))
    
    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer= 'adam',loss='binary_crossentropy',metrics=['accuracy'])
    
    return model

In [ ]:
# Create a KerasClassifier
model = KerasClassifier(layers=1,neurons=32,build_fn=create_model,epochs=50, batch_size=10, verbose=0)

In [ ]:
# Define grid search parameters
param_grid = {
    'neurons':[16,32,64,128],
    'layers': [1,2,3],
    # 'batch_size': [10,20]
    'epochs': [50,100]
}

In [19]:
# performing grid search 
grid = GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=-1,cv=3)
grid_result = grid.fit(X_train,y_train)
print("Best: %f using %s" % (grid_result.best_score_,grid_result.best_params_))

c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\sharm\anaconda3\envs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best: 0.857000 using {'epochs': 50, 'layers': 1, 'neurons': 64}
